# 💧 LFM3-Audio - Voice commands to function calls with Transformers

This tutorial demonstrates how to fine-tune [`LiquidAI/LFM3-Audio-2B`](https://huggingface.co/LiquidAI/LFM3-Audio-2B) to map **spoken smart-home
commands directly to function calls**: audio in, function call out, with no separate speech-to-text step.

```
🎙️ "turn on the fan in the living room"   →   HassTurnOn|$area=living room|$domain=fan
```

We use the same task and data as our [LFM2.5-Audio voice assistant example](https://github.com/Liquid4All/cookbook/tree/main/examples/voice-assistant):
[OHF-Voice](https://huggingface.co/datasets/Paulescu/OHF-Voice-audio-20260504), 55k recorded Home Assistant commands
paired with their function call, across 41 functions (timers, lights, climate, media, shopping lists, ...).

## 🎯 What You'll Find:
- **Data preparation** - (audio, function call) pairs as chat conversations
- **Baseline** - what the model does before fine-tuning
- **LoRA + SFT** - parameter-efficient fine-tuning with the 🤗 `Trainer`
- **Evaluation** - format compliance, function-name accuracy and exact-match accuracy on a held-out set

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type` (an A100 is recommended)
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages:

In [ ]:
!pip install -qqq "transformers==5.20.2" "torchao==0.18.0" accelerate datasets peft librosa soundfile --progress-bar off

Let's now verify the packages are installed correctly

In [ ]:
import os
import torch
import transformers
import peft
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"🧩 PEFT version: {peft.__version__}")

# Loading the model from Transformers 🤗

We only need text output here; the processor also loads the audio codec, but this tutorial never uses it.

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Audio-2B"

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

print("🧠 Loading model...")
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True
)

print("\n✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

# 📚 Load the dataset

The full training split has 52k samples in 16 shards of ~470 MB. Two shards (~6,600 samples) are enough to learn the task;
add more for better accuracy on rare functions. For evaluation we draw up to 10 samples per function from the test split
(~400 samples), as in the LFM2.5 example.

Each sample holds the recording (WAV bytes), the function call, and the transcript of the command (which we only use for display).

In [ ]:
import collections
import io
import random

import librosa
import soundfile as sf
from datasets import load_dataset
from IPython.display import Audio, display

TRAIN_SHARDS = 2
dataset_id = "Paulescu/OHF-Voice-audio-20260504"
train_files = [f"data/train-{i:05d}-of-00016.parquet" for i in range(TRAIN_SHARDS)]
# verification_mode="no_checks": we load a subset of the files, which doesn't match the split sizes in the dataset card
train_raw = load_dataset(dataset_id, data_files={"train": train_files}, split="train", verification_mode="no_checks")
test_raw = load_dataset(dataset_id, data_files={"test": "data/test-00000-of-00001.parquet"}, split="test", verification_mode="no_checks")


def flatten(row):
    return {
        "wav": row["audio_chat"][0]["content"][0]["audio"],
        "target": row["audio_chat"][1]["content"][0]["text"],
        "transcript": row["text_chat"][0]["content"][0]["text"],
    }


train_raw = train_raw.map(flatten, remove_columns=train_raw.column_names)
test_raw = test_raw.map(flatten, remove_columns=test_raw.column_names)

# stratified evaluation subset: up to 10 samples per function
by_function = collections.defaultdict(list)
for i, target in enumerate(test_raw["target"]):
    by_function[target.split("|")[0]].append(i)
rng = random.Random(0)
eval_raw = test_raw.select(sorted(i for idx in by_function.values() for i in rng.sample(idx, min(10, len(idx)))))

print("✅ Dataset loaded:")
print(f"   📚 Train samples: {len(train_raw)}")
print(f"   🧪 Eval samples: {len(eval_raw)} across {len(by_function)} functions")
for row in train_raw.select(range(3)):
    print(f"\n🎙️ {row['transcript']!r}\n🔧 {row['target']}")
    display(Audio(row["wav"]))

## Format the conversations

The recording is the whole user turn; the function call is the assistant's answer. A short system prompt names the task.
Recordings are 24 kHz; we resample them to the 16 kHz the audio encoder expects.

In [ ]:
SYSTEM_PROMPT = "Convert the spoken command into a Home Assistant function call."


def load_16k(wav_bytes):
    wav, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    return librosa.resample(wav, orig_sr=sr, target_sr=16_000)


def format_sample(row, with_answer=True):
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "audio", "audio": load_16k(row["wav"])}]},
    ]
    if with_answer:
        conversation.append({"role": "assistant", "content": row["target"]})
    return conversation


print(processor.apply_chat_template(format_sample(train_raw[0]), tokenize=False))

## Collate function

The processor renders a batch of conversations, computes the log-mel features of each recording and expands every
`<|audio|>` placeholder to its audio embeddings. We train only on the answer: every token up to and including the
assistant header is masked with `-100`. Recordings are decoded on the fly, so the dataset stays small in memory.

In [ ]:
_prompt = processor.apply_chat_template(format_sample(train_raw[0], with_answer=False), add_generation_prompt=True, tokenize=False)
ASSISTANT_HEADER = processor.tokenizer(_prompt[_prompt.rindex("<|im_start|>"):], add_special_tokens=False)["input_ids"]


def answer_start(ids):
    n = len(ASSISTANT_HEADER)
    return max(i for i in range(len(ids) - n + 1) if ids[i:i + n] == ASSISTANT_HEADER) + n


def collate_fn(rows):
    batch = processor.apply_chat_template(
        [format_sample(row) for row in rows], tokenize=True, return_dict=True, return_tensors="pt", processor_kwargs={"padding": True}
    )
    labels = torch.full_like(batch["input_ids"], -100)
    for i, ids in enumerate(batch["input_ids"].tolist()):
        start = answer_start(ids)
        labels[i, start:] = batch["input_ids"][i, start:]
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch


batch = collate_fn([train_raw[0], train_raw[1]])
print({k: tuple(v.shape) for k, v in batch.items()})
print("🎯 Supervised tokens:", [processor.tokenizer.decode(ids[lab != -100]) for ids, lab in zip(batch["input_ids"], batch["labels"])])

# 📏 Baseline

We score predictions with three layered metrics, as in the LFM2.5 example:
- **Format**: the output parses as `Function|$key=value|...`
- **Function**: the function name is right
- **Exact match**: the function name and every argument are right (strict string equality)

`suppress_mixed=True, suppress_audio_out=True` keep the answer in text.

In [ ]:
import re
from tqdm.auto import tqdm


def parse(call):
    parts = call.strip().split("|")
    if not re.fullmatch(r"[A-Za-z]+", parts[0]) or any(not re.fullmatch(r"\$[^=|]+=.*", p) for p in parts[1:]):
        return None
    return parts[0], dict(p[1:].split("=", 1) for p in parts[1:])


@torch.inference_mode()
def evaluate(model, show=5):
    scores = collections.Counter()
    for k, row in enumerate(tqdm(eval_raw, desc="Evaluating")):
        inputs = processor.apply_chat_template(
            format_sample(row, with_answer=False),
            add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
        ).to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, suppress_mixed=True, suppress_audio_out=True)
        prediction = processor.decode(out.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if k < show:
            print(f"   🎙️ {row['transcript']!r}\n      → {prediction!r}\n      ✅ {row['target']!r}")
        predicted, expected = parse(prediction), parse(row["target"])
        scores["format"] += predicted is not None
        scores["function"] += predicted is not None and predicted[0] == expected[0]
        scores["exact"] += predicted == expected
    return {metric: scores[metric] / len(eval_raw) for metric in ("format", "function", "exact")}


def report(name, scores):
    print(f"{name:8s} format {scores['format']:6.1%}   function {scores['function']:6.1%}   exact match {scores['exact']:6.1%}")


scores_before = evaluate(model)
report("before", scores_before)

Without fine-tuning, the model has never seen this function vocabulary or the pipe-delimited format, so it typically just transcribes the command.
Fine-tuning bakes the 41 function signatures into the weights.

## Wrap the model with PEFT

We add LoRA adapters to the language model: the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). The audio encoder stays frozen. LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full fine-tuning
- Lower → needs even less compute resources

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # optional: add "|gate_proj" to the group for LFM3's per-head attention gate
    target_modules=r"model\.language_model\..*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|w1|w2|w3)",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 🚀 Launch Training

We use the 🤗 `Trainer`, which trains on the loss returned by the model. Our collate function builds the batches, so we keep all
columns (`remove_unused_columns=False`). One epoch over two shards takes ~10 minutes on an A100.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="lfm3-audio-voice-commands",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=20,
    save_strategy="no",
    bf16=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
)

print("🏗️  Creating trainer...")
trainer = Trainer(model=model, args=training_args, train_dataset=train_raw, data_collator=collate_fn)

print("\n🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"💾 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# 📈 Evaluate the fine-tuned model

In [ ]:
model.eval()
scores_after = evaluate(model)
report("before", scores_before)
report("after", scores_after)

# 💾 Save the model

Save the LoRA adapter, or merge it into the base weights to obtain a regular checkpoint.

In [ ]:
model.save_pretrained("./lfm3-audio-voice-commands-lora")
print("💾 Adapter saved to: ./lfm3-audio-voice-commands-lora")

In [ ]:
print("🔄 Merging LoRA weights...")
merged = model.merge_and_unload()
merged.save_pretrained("./lfm3-audio-voice-commands")
processor.save_pretrained("./lfm3-audio-voice-commands")
print("💾 Model saved to: ./lfm3-audio-voice-commands")

# 🚀 Next steps

- **More data**: raise `TRAIN_SHARDS` (up to 16) to improve accuracy on rare functions and argument values.
- **Your own commands**: record your users' phrasings, or synthesize them with the model's own text-to-speech, and pair them with your API's function calls.
- **Deploy on device**: the [LFM2.5 voice assistant example](https://github.com/Liquid4All/cookbook/tree/main/examples/voice-assistant) shows how to quantize the fine-tuned model to GGUF and serve it with llama.cpp.
- [`lfm3_audio_tts_sft_with_transformers.ipynb`](lfm3_audio_tts_sft_with_transformers.ipynb) teaches the model a new voice, training its speech output.